# Private RAG Agent — AMD Radeon GPU Benchmark

本笔记本用于在 Radeon Cloud 实例上采集 **AMD 40 分证据**（ROCm 环境、GPU 状态、推理吞吐）。
**逐格运行或 Kernel → Restart & Run All**，把每格输出截图存到 `benchmarks/`。

## ① ROCm 环境证据

In [ ]:
!echo '==== rocminfo ====' && rocminfo 2>/dev/null | head -40 || echo 'rocminfo not found'

In [ ]:
!echo '==== rocm-smi ====' && rocm-smi 2>/dev/null || echo 'rocm-smi not found'
!echo && echo '==== GPU vendor ====' && lspci 2>/dev/null | grep -iE 'vga|display|amdgpu' | head -5 || true

## ② 检查运行中的推理服务

In [ ]:
import subprocess
r = subprocess.run('ps aux | grep -E "llama|vllm|ollama" | grep -v grep', shell=True, capture_output=True, text=True)
print('==== running inference processes ====')
print(r.stdout[:2000] if r.stdout.strip() else '(none)')

In [ ]:
import socket
for port in [8000, 8080, 11434, 8888, 5001]:
    s = socket.socket(); s.settimeout(0.5)
    try:
        s.connect(('127.0.0.1', port)); print(f'port {port}: OPEN')
    except Exception:
        print(f'port {port}: closed')
    finally:
        s.close()

## ③ 推理吞吐基准

若检测到 OpenAI 兼容服务（vLLM/llama-server，通常 8000/8080 端口），直接测量 tokens/s。
若没有，请先运行同目录 `hermes-agent-vllm-local-serving-radeon-workshop.ipynb`（官方 workshop）启动服务，再回来跑本格。

In [ ]:
import json, time, urllib.request

def probe(port):
    try:
        req = urllib.request.Request(f'http://127.0.0.1:{port}/v1/models', headers={'Content-Type':'application/json'})
        with urllib.request.urlopen(req, timeout=3) as resp:
            return json.loads(resp.read().decode())
    except Exception:
        return None

found = None
for port in [8000, 8080, 11434]:
    m = probe(port)
    if m and m.get('data'):
        found = port
        print(f'found serving on :{port}  model={[x["id"] for x in m["data"]][:3]}')
        break
print('serving port:', found)

In [ ]:
import json, time, urllib.request

PORT = 8000  # 检测到的服务端口；若上格输出了别的端口，改成那个
PROMPTS = ['什么是 RAG？请用三句话回答。', '用三句话介绍 AMD ROCm。', '简述混合检索与重排。']

def chat(prompt, max_tokens=256):
    body = json.dumps({'model':'model','messages':[{'role':'user','content':prompt}],'max_tokens':max_tokens,'temperature':0.3}).encode()
    req = urllib.request.Request(f'http://127.0.0.1:{PORT}/v1/chat/completions', data=body, headers={'Content-Type':'application/json'})
    with urllib.request.urlopen(req, timeout=180) as resp:
        return json.loads(resp.read().decode())

results = []
for q in PROMPTS:
    try:
        t0 = time.time()
        r = chat(q)
        dt = time.time() - t0
        usage = r.get('usage', {})
        n = usage.get('completion_tokens') or max(len(r['choices'][0]['message']['content'])//2, 1)
        tps = n / dt if dt > 0 else 0
        results.append((q[:18], n, round(dt,2), round(tps,1)))
        print(f'{results[-1][0]:20s} tokens={n:4d} time={dt:6.2f}s  {tps:6.1f} tok/s')
    except Exception as e:
        print(f'{q[:18]:20s} ERR {str(e)[:80]}')

if results:
    avg = sum(r[3] for r in results)/len(results)
    print(f'
==== 平均吞吐: {avg:.1f} tokens/s (AMD Radeon GPU / ROCm) ====')

## ④ 截图清单（存到本机 `benchmarks/`）
- 第①部分：`rocminfo` 头部 + `rocm-smi` 输出 → **ROCm 环境证据**
- 第③部分：tokens/s 汇总 → **推理吞吐对比**
- 可选：浏览器打开 workshop 服务地址，展示模型在 AMD GPU 上实时推理

> 结果填进 README 性能表后，PR #154 会自动同步更新。